# Silverwing-ML: GPU Training

1. Enable GPU: Runtime → Change runtime type → T4 GPU
2. Run cells in order
3. Upload the zip when prompted (or upload to Drive first)

In [ ]:
# Cell 1: Check GPU
!nvidia-smi
import torch
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# Cell 2: Install dependencies
!pip install -q torch --index-url https://download.pytorch.org/whl/cu121
!pip install -q pyyaml numpy
print('Done')

In [ ]:
# Cell 3: Upload silverwing-colab.zip from your computer
# If this fails with a JSON error, upload to Google Drive instead
# and change the path below to '/content/drive/MyDrive/silverwing-colab.zip'
import os

# --- TRY DRIVE FIRST ---
DRIVE_ZIP = '/content/drive/MyDrive/silverwing-colab.zip'
LOCAL_ZIP = '/content/silverwing-colab.zip'

zip_path = None

# Check if already on Drive
if os.path.exists(DRIVE_ZIP):
    zip_path = DRIVE_ZIP
    print(f'Found zip on Drive: {DRIVE_ZIP}')
else:
    # Try uploading from browser
    try:
        from google.colab import files
        print('Upload silverwing-colab.zip:')
        uploaded = files.upload()
        for name in uploaded:
            zip_path = f'/content/{name}'
            break
    except Exception as e:
        print(f'Upload failed: {e}')
        print('Falling back to Drive mount...')

# Mount Drive as fallback
if zip_path is None:
    from google.colab import drive
    drive.mount('/content/drive')
    if os.path.exists(DRIVE_ZIP):
        zip_path = DRIVE_ZIP
        print(f'Found zip on Drive: {DRIVE_ZIP}')
    else:
        raise FileNotFoundError(
            f'Cannot find silverwing-colab.zip.\n'
            f'Upload it to: {DRIVE_ZIP}\n'
            f'Or upload directly when prompted.'
        )

!unzip -q $zip_path -d /content/
os.chdir('/content/Silverwing-ML')
print(f'Project at: {os.getcwd()}')

In [ ]:
# Cell 4: Mount Drive for checkpoint storage
import os
try:
    from google.colab import drive
    drive.mount('/content/drive')
except:
    pass  # already mounted

DRIVE_DIR = '/content/drive/MyDrive/silverwing'
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f'Checkpoints save to: {DRIVE_DIR}')

In [ ]:
# Cell 5: Pretrain (saves to Drive)
!python scripts/train.py \
    --config configs/training.yaml \
    --device cuda \
    --max-steps 5000 \
    --batch-size 4 \
    --checkpoint-dir $DRIVE_DIR/pretrain \
    --no-clean-repo-check

In [ ]:
# Cell 6: SFT (saves to Drive)
!python scripts/train_sft.py \
    --config configs/sft_combined.yaml \
    --init-from $DRIVE_DIR/pretrain/best.pt \
    --device cuda \
    --checkpoint-dir $DRIVE_DIR/sft-combined \
    --no-clean-repo-check

In [ ]:
# Cell 7: List checkpoints on Drive
import os
print('=== Checkpoints ===')
for root, dirs, files in os.walk(DRIVE_DIR):
    for f in sorted(files):
        if f.endswith('.pt'):
            size_mb = os.path.getsize(os.path.join(root, f)) / 1e6
            print(f'{f} ({size_mb:.0f} MB)')
print(f'All at: {DRIVE_DIR}')